# Model usage and evaluation

This notebook provides a simple way of using the already trained model to make predictions on the test set and aims at analysing the results. We draw simple conclusions on the model's performance + what could be done to improve it.

#### Libs and dependancies

Note: create a virtual environment and reload the notebbok's kernel if needed

In [16]:
!pip install -q -r requirements.txt
!python -m spacy download xx_sent_ud_sm

  Using cached https://github.com/explosion/spacy-models/releases/download/xx_sent_ud_sm-3.8.0/xx_sent_ud_sm-3.8.0-py3-none-any.whl (4.3 MB)
✔ Download and installation successful
You can now load the package via spacy.load('xx_sent_ud_sm')


In [17]:
# Our own data processing tools
from data_procesing import load_raw_data, EOSDataset, prepare_text

# Other libaries
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch
from sklearn.metrics import classification_report

---

### 1. Loading the fine-tuned model

If necessary, the model can be trained again by running the ```finetuning.py``` script, which automatically stores the classification head's weights in the correct folder. This is normally not needed as trained weights are already provided

In [18]:
model_name = "bert-base-multilingual-cased"
model = AutoModelForTokenClassification.from_pretrained(model_name, num_labels=2)
model.classifier.load_state_dict(torch.load("weights/eos_head.pt"))
model.eval()

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 2771.11it/s]
BertForTokenClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok

BertForTokenClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(119547, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, e

### 2. Loading the test set

In [19]:
tokenizer = AutoTokenizer.from_pretrained('bert-base-multilingual-cased')

raw_test = load_raw_data('test')
test_set = EOSDataset(raw_test,tokenizer)
test_loader = DataLoader(test_set,batch_size=16,shuffle='True')

Reading : en_pud-ud-test.sent_split
Reading : en_partut-ud-test.sent_split
Reading : it_vit-ud-test.sent_split
Reading : en_gum-ud-test.sent_split
Reading : it_partut-ud-test.sent_split
Reading : it_markit-ud-test.sent_split
Reading : en_ewt-ud-test.sent_split
Reading : it_isdt-ud-test.sent_split


### 3. Making our first predictions

In [21]:
all_preds, all_true = [], []

with torch.no_grad():
    for batch in test_loader:
        outputs = model(
            input_ids = batch["input_ids"],
            attention_mask = batch["attention_mask"],
        )

        preds = outputs.logits.argmax(dim=-1)

        # Flatten the obtained results and excluding padding tokens
        mask = batch["labels"] != -100
        all_preds.extend(preds[mask].tolist())
        all_true.extend(batch["labels"][mask].tolist())

In [22]:
print(classification_report(all_true, all_preds, target_names=["non-EOS", "EOS"]))

              precision    recall  f1-score   support

     non-EOS       1.00      0.98      0.99      5911
         EOS       0.61      0.99      0.76       201

    accuracy                           0.98      6112
   macro avg       0.81      0.98      0.87      6112
weighted avg       0.99      0.98      0.98      6112



**Analysis:** First of all, the near perfect acuracy in our case is irrelevant due to class imbalance. We have way more non-EOS tokens and the real interesting part is analysing precision/recall for the positive class.

We see that we achieve a very good Recall score, meaning our model does not miss a lot of real sentence endings. However we get a quite poor precision score, meaning our model tends to be very conservative and labels sentence endings that do not exist in reality (A lot of False Positives).

This can be due to the dataset structure: we know some <EOS> in the original dataset are not really sentence in the strictest sense but more end of titles, sections, etc. Which can explain this bias.

To get a clearer picture we can visualize which sentences were predicted by our model:  
(for transparency: AI coded this)

In [30]:
def reconstruct_sentences(text_raw, tokenizer, model):
    encoded = prepare_text(text_raw, tokenizer)
    
    input_ids      = torch.tensor([encoded["input_ids"]])
    attention_mask = torch.tensor([encoded["attention_mask"]])

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    
    preds = outputs.logits.argmax(dim=-1)[0]

    # Regroupe les input_ids entre chaque EOS prédit
    current = []
    sentences = []
    for token_id, pred in zip(encoded["input_ids"], preds.tolist()):
        current.append(token_id)
        if pred == 1:
            sentences.append(tokenizer.decode(current, skip_special_tokens=False))
            current = []
    
    if current:
        sentences.append(tokenizer.decode(current, skip_special_tokens=False))
    
    return sentences

In [31]:
sentences = reconstruct_sentences(raw_test[0], tokenizer, model)
for i, s in enumerate(sentences):
    print(f"{i+1}. {s}")

1. [CLS] [UNK]
2. While much of the digital transition is unprecedented in the United States, the peaceful transition of power is not,
3. [UNK] Obama special assistant Kori Schulman wrote in a blog post Monday.
4. For those who follow social media transitions on Capitol Hill, this will be a little different.
5. But in a break from his past rhetoric about curtailing immigration, the GOP nominee proclaimed that as president he would allow [UNK] tremendous numbers [UNK] of legal immigrants based on a [UNK] merit system.
6. [UNK]
7. [UNK]
8. So I hate to put a little pressure on you,
9. but the fate of the republic rests on your shoulders,
10. [UNK] he told the crowd gathered on a sports field at the University of North Carolina.
11. The new spending is fueled by Clinton [UNK] s large bank account.
12. What she [UNK] s saying and what she [UNK] s doing, it [UNK] actually, it [UNK] s unbelievable.
13. $ 5, 000 per person, the maximum allowed.
14. In early October, the transition team used t

Here we clearly see that a lot of the time, our model interprets the [UNK] token as an end of sentence marker. By looking up in the original text we can see that it corresponds to a variety of situations and mainly to citations (ex: The president said: "I love this country.") which explains the confusion as brackets often appear after the end of sentence punctuation...

This isn't the only explanation of course, but this little overview tells us why and how our Precision score is low.

**Possible solutions identified:**
1. Train the model further. I have the intuition that maybe unfreezing the base model would allow it to understand the context more deeply and avoid this type of error. This is pure speculation i have not tested it as my PC already struggles to train just the classification head.

2. Adjust the decision treshold. This is an easy fix, classifying as positive only tokens with a high predicted probability (instead of regular 1/2 decision boundary). This guarantees automatically better precision (but maybe lower recall). We choose to do this for the sake of simplicity.

### 4. Adjusting the decision treshold 

Following what we said, we make new predictions using a probability decision treshold set to 0.7

In [36]:
all_preds, all_true = [], []

with torch.no_grad():
    for batch in test_loader:
        outputs = model(
            input_ids = batch["input_ids"],
            attention_mask = batch["attention_mask"],
        )

        probs = torch.softmax(outputs.logits, dim=-1) # Taking just the probabilities
        eos_probs = probs[:, :, 1]
        preds = (eos_probs > 0.7).long() # New treshold

        # Flatten the obtained results and excluding padding tokens
        mask = batch["labels"] != -100
        all_preds.extend(preds[mask].tolist())
        all_true.extend(batch["labels"][mask].tolist())

In [37]:
print(classification_report(all_true, all_preds, target_names=["non-EOS", "EOS"]))

              precision    recall  f1-score   support

     non-EOS       1.00      0.99      1.00      5911
         EOS       0.79      0.98      0.87       201

    accuracy                           0.99      6112
   macro avg       0.89      0.98      0.93      6112
weighted avg       0.99      0.99      0.99      6112



Here the results looks much more promising: We get a significant increase in precision and abrely lost any recall. Rejecting the weaker predicted probabilities helps eliminating a lot of the false positives. This gives us, in turn, an musch more honorable F1-score.

Let's now compare to the performance of a multilingual model in Spacy

### 5. Comparison with Spacy

We use here a multilingual Spacy model for fair comparison. We also need to compute again the original character-to-token offest to be able to compare both models outputs.  
(For transparency again: AI helped a lot in this)

In [38]:
import spacy
import re
nlp = spacy.load("xx_sent_ud_sm")

all_true, all_bert, all_spacy = [], [], []

for text_raw in raw_test:
    clean   = re.sub(r'<EOS>', '', text_raw)
    encoded = prepare_text(text_raw, tokenizer)
    offsets = encoded["offset_mapping"]
    
    # Our model predictions
    with torch.no_grad():
        outputs = model(
            input_ids=      torch.tensor([encoded["input_ids"]]),
            attention_mask= torch.tensor([encoded["attention_mask"]])
        )
    probs = torch.softmax(outputs.logits, dim=-1)[0, :, 1] 
    bert_preds = (probs > 0.7).long().tolist() # We keep our previous treshold
    
    # Spacy predicting which character ends a sentence
    spacy_ends = {sent.end_char for sent in nlp(clean).sents}
    
    # Looping on the original offsets
    for i, (start, end) in enumerate(offsets):
        if end == 0:  # ignoring special tokens
            continue
        all_true.append(encoded["labels"][i])
        all_bert.append(bert_preds[i])
        all_spacy.append(1 if end in spacy_ends else 0)

print("=== Our model ===")
print(classification_report(all_true, all_bert, target_names=["non-EOS", "EOS"]))
print("=== Spacy ===")
print(classification_report(all_true, all_spacy, target_names=["non-EOS", "EOS"]))

=== Our model ===
              precision    recall  f1-score   support

     non-EOS       1.00      0.99      1.00      3948
         EOS       0.79      0.97      0.87       132

    accuracy                           0.99      4080
   macro avg       0.89      0.98      0.93      4080
weighted avg       0.99      0.99      0.99      4080

=== Spacy ===
              precision    recall  f1-score   support

     non-EOS       0.99      1.00      1.00      3948
         EOS       0.96      0.76      0.85       132

    accuracy                           0.99      4080
   macro avg       0.98      0.88      0.92      4080
weighted avg       0.99      0.99      0.99      4080



We see that our model here barely beats Spacy regarding F1-score, mission accomplished :)

It is interesting to see that Spacy has the inverse problem of our model: It tends to have better precision but misses quite a lot of <EOS> with a lower recall. This can be attributed to the dataset which contains end-of-sentences that are grammatically incorrect (e.g end of a title without punctuation.)

A contrario, our model correctly identifies this kind of cases but tends to over-generalize it to the rest of the dataset, somtimes cutting after a comma for exemple (as seen in the exemple above).

#### Conclusion

Our model needs more training to better discern the edge cases. It is clear that the next step in our approach is unfreezing the base encoder to allow for a deeper understanding of the data structure. I imagine that feeding it more training data would also help. Maybe using more advanced finetuning techniques could also help (as seen in the last lecture).

In the end there is a lot of things we could try/do but I am satified with the currently obtained result!